In [188]:
import os
import json
import snowflake.connector
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings("ignore")

In [189]:
TOPICS = ["food quality", "delivery", "pricing", "service", "packaging", "other"]

SYSTEM_PROMPT = f"""
You are the expert in detecting the sentiment of the reviews posted by various customers. A review posted by the customer is given. Your job is to detect the following:

- sentiment-label: Mention whether the review is Positive or Negative or Neutral.
- sentiment-score: Retrieve the sentiment score for the review between -1 and 1.
- topic: Mention under which of the following topics this review comes under
{TOPICS}
- key_issue: a short phrase of 6 words or less that describes the main issue in the review, if any. If there is no issue, return null

Reply as JSON in this exact format:
{{
    "sentiment_label": "<sentiment_label>",
    "sentiment_score": <sentiment_score>,
    "topic": "<topic>",
    "key_issue": "<key_issue>"
}}
"""

In [190]:
help(snowflake.connector)

Help on package snowflake.connector in snowflake:

NAME
    snowflake.connector - PEP 249 Database API 2.0 Implementation

DESCRIPTION
    This module provides an empty implementation of the Python Database API Specification 2.0
    as defined in PEP 249.

PACKAGE CONTENTS
    _common (package)
    _internal (package)
    aio (package)
    auth (package)
    compat
    config_manager
    connection (package)
    connection_config
    constants
    cursor (package)
    description
    errors
    network
    pandas_tools
    result_batch
    telemetry
    time_util
    types
    util_text
    version

SUBMODULES
    _core

CLASSES
    builtins.Exception(builtins.BaseException)
        snowflake.connector.errors.Error
            snowflake.connector.errors.DatabaseError
                snowflake.connector.errors.DataError
                snowflake.connector.errors.IntegrityError
                snowflake.connector.errors.InternalError
                snowflake.connector.errors.NotSupporte

In [191]:
def get_connection():
    conn = snowflake.connector.connect(
        user=os.getenv("SNOWFLAKE_USER"),
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        role=os.getenv("SNOWFLAKE_ROLE"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
    )
    return conn

In [192]:
def create_review_enriched_table(cursor):
    cursor.execute("CREATE SCHEMA IF NOT EXISTS ZOMATO.AI")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ZOMATO.AI.REVIEW_ENRICHED (
            REVIEW_ID STRING,
            SENTIMENT_LABEL STRING,
            SENTIMENT_SCORE FLOAT,
            TOPIC STRING,
            KEY_ISSUE STRING,
            MODEL STRING,
            ENRICHED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
        )
    """)

In [193]:
def retrieve_reviews(cursor):
    cursor.execute("SELECT REVIEW_ID, COMMENT FROM ZOMATO.RAW.REVIEWS limit 50")
    return cursor.fetchall()

In [194]:
from openai import OpenAI

client = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1"
)

def classify_review(comment):
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": comment}
        ],
        temperature=0.5,
        response_format={"type": "json_object"},
        model="lfm3-omni:latest"
    )

    return response.choices[0].message.content

In [195]:
def insert_into_reviews_enriched(cursor,result):
    """Insert all the enriched rows into Snowflake in one go."""
    print(f"Saving 5 enriched reviews to Snowflake...")
    cursor.executemany(
        """
        INSERT INTO ZOMATO.AI.REVIEW_ENRICHED
            (REVIEW_ID, SENTIMENT_LABEL, SENTIMENT_SCORE, TOPIC, KEY_ISSUE, MODEL)
        VALUES (%s, %s, %s, %s, %s, %s)
        """,
        result
    )

In [196]:
def main():
    conn = get_connection()
    cursor = conn.cursor()
    create_review_enriched_table(cursor)
    reviews = retrieve_reviews(cursor)
    result = []
    it = 0
    for ctr,review in reviews:
        t = []
        t.append(ctr)
        p = json.loads(classify_review(review))
        for j in p:
            t.append(p[j])
        t.append("lfm3-omni:latest")
        result.append(tuple(t))
        it += 1
        print(it,":",ctr)
    print(result)
    insert_into_reviews_enriched(cursor,result)
    conn.commit()
    cursor.close()
    conn.close()

In [197]:
main()

1 : 1
2 : 2
3 : 3
4 : 4
5 : 5
6 : 6
7 : 7
8 : 8
9 : 9
10 : 10
11 : 11
12 : 12
13 : 13
14 : 14
15 : 15
16 : 16
17 : 17
18 : 18
19 : 19
20 : 20
21 : 21
22 : 22
23 : 23
24 : 24
25 : 25
26 : 26
27 : 27
28 : 28
29 : 29
30 : 30
31 : 31
32 : 32
33 : 33
34 : 34
35 : 35
36 : 36
37 : 37
38 : 38
39 : 39
40 : 40
41 : 41
42 : 42
43 : 43
44 : 44
45 : 45
46 : 46
47 : 47
48 : 48
49 : 49
50 : 50
[(1, 'Positive', 0.9, 'packaging', None, 'lfm3-omni:latest'), (2, 'Negative', -0.9, 'pricing', 'Too expensive', 'lfm3-omni:latest'), (3, 'Neutral', 0.0, 'other', None, 'lfm3-omni:latest'), (4, 'Positive', 0.6, 'delivery', None, 'lfm3-omni:latest'), (5, 'Positive', 0.6, 'packaging', None, 'lfm3-omni:latest'), (6, 'Negative', -0.9, 'food quality', 'spilled gravy', 'lfm3-omni:latest'), (7, 'Positive', 0.9, 'packaging', None, 'lfm3-omni:latest'), (8, 'Negative', -0.9, 'pricing', 'oily, stale and tasteless', 'lfm3-omni:latest'), (9, 'Neutral', 0.0, 'other', None, 'lfm3-omni:latest'), (10, 'Positive', 0.9, 'food qual

In [281]:
import os
import numpy as np
import pandas as pd
import snowflake.connector
from openai import OpenAI
from dotenv import load_dotenv
import streamlit as st

In [282]:
CACHE_FILE = "review_embeddings.parquet"
EMBEDDING_MODEL = "nomic-embed-text"
CHATBOT_MODEL = "lfm3-omni:latest"
SAMPLE_RATE = 500
TOP_K = 5

In [283]:
client = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1"
)

In [284]:
def convert_to_embeddings(reviews):
    embed = client.embeddings.create(
        input=reviews,
        model=EMBEDDING_MODEL
    )
    pt = [i.embedding for i in embed.data]
    return pt

In [285]:
def get_connection():
    conn = snowflake.connector.connect(
        user=os.getenv("SNOWFLAKE_USER"),
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        role=os.getenv("SNOWFLAKE_ROLE"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
    )
    return conn

In [288]:
def load_reviews():
    conn = get_connection()
    cursor = conn.cursor()
    rows = cursor.execute(f"SELECT REVIEW_ID, CITY, RATING, COMMENT FROM ZOMATO.STAGING.STG_REVIEWS SAMPLE({SAMPLE_RATE} ROWS)")
    df = rows.fetch_pandas_all()
    df.columns = [i.lower() for i in df.columns]
    return df

In [289]:
df = load_reviews()
df.head()

,review_id,city,rating,comment
0,37499,Indore,4,"Delivered right on time. Smooth, hassle-free e..."
1,85669,Kanpur,1,Got the wrong order entirely. Overpriced and n...
2,22690,Delhi,1,The food was cold and bland. Arrived way too l...
3,157057,Kolkata,3,An average experience overall.
4,233470,Tuni,3,Decent but could be better.


In [302]:
@st.cache_data()
def load_cache():
    if os.path.exists(CACHE_FILE):
        df = pd.read_parquet(CACHE_FILE)
        return df
    else:
        df = load_reviews()
        df["embeddings"] = convert_to_embeddings(df["comment"].tolist())
        df.to_parquet(CACHE_FILE)
        return df

2026-11-18 18:19:30.848 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [303]:
a = [1,2,3]
b = [4,5,6]
np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b))

np.float64(0.9746318461970762)

In [304]:
np.sqrt(1*1+2*2+3*3)

np.float64(3.7416573867739413)

In [305]:
def cosine_similarity(a,b):
    return np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b))

In [306]:
def retrieve_top_rows_with_similarity_search(question,df):
    question_embedding = convert_to_embeddings(question)
    sim = []
    for i in df["embeddings"].to_list():
        review_embedding = i.copy()
        sim.append(cosine_similarity(question_embedding,review_embedding))
    df["similarity_score"] = sim
    df = df.sort_values(by="similarity_score").reset_index(drop=True).head(TOP_K)
    return df

In [310]:
def chatbot(question,df):
    reviews = ""
    SYSTEM_PROMPT = "Answer ONLY using the customer reviews provided.\nBe concise. If the reviews don't covert it, say so"
    for i in range(df.shape[0]):
        reviews += f" ({df.loc[i,'city']}, {df.loc[i,'rating']} stars) {df.loc[i,'comment']}\n"
    USER_PROMPT = f"Question: {question}\n\nTop 5 reviews:\n{reviews}"

    response = client.chat.completions.create(
        messages=[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":USER_PROMPT}
        ],
        model=CHATBOT_MODEL,
        temperature=0.5
    )
    return response.choices[0].message.content

In [311]:
question = "Top 5 restaurant reviews"
df = load_cache()
df_1 = retrieve_top_rows_with_similarity_search(question,df)
df_2 = chatbot(question,df_1)

'The top 5 restaurant reviews are all 1 or 2 stars, and all say the food is "far too expensive for what you get."'